### Description
Loads cleaned parquets for Track B datasets (depmap_expr, hpa_rna, hpa_desc, geo_expr), verifies candidate gene and cell line keys against shared lookup tables, measures match/overlap rates, and surfaces unresolved or duplicate identities for handoff to 03_data_integration.


depmap_expr, hpa_rna, geo_expr --> all three need GENE key harmonisation (ENSG) and all three need CELL LINE key resolution

hpa_desc -- > exists specifically to help resolve hpa_rna's cell line identity gaps (it has CVCL accessions that hpa_rna itself lacks)

### Confirm gene key format across all three expression files

In [9]:
import pandas as pd
import re

depmap = pd.read_parquet("../../data/parquet/data_clean/depmap_expr_clean.parquet")
hpa    = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
geo    = pd.read_parquet("../../data/parquet/data_clean/geo_expr_clean.parquet")

# depmap_expr — gene is in COLUMN HEADERS, already cleaned by
# clean_depmap_expr() to extract bare ensg from "symbol (ENSGxxx)"
depmap_genes = set(c for c in depmap.columns if re.match(r"^ensg\d+$", str(c)))
print(f"depmap_expr: {len(depmap_genes)} clean ENSG columns")
print(f"depmap_expr: {len(depmap.columns) - len(depmap_genes)} columns did NOT match clean ENSG pattern")
non_ensg_cols = [c for c in depmap.columns if c not in depmap_genes]
print(f"Sample non-ENSG columns: {non_ensg_cols[:10]}")

# hpa_rna — gene is a ROW VALUE
hpa_genes = set(hpa["gene"].dropna())
bad_hpa = hpa[~hpa["gene"].astype(str).str.match(r"^ensg\d+$", na=False)]
print(f"\nhpa_rna: {len(hpa_genes)} unique gene values")
print(f"hpa_rna: {len(bad_hpa)} rows with non-clean ENSG format")

# geo_expr — gene is also a ROW VALUE (still wide, gene per row)
geo_genes = set(geo["gene"].dropna())
bad_geo = geo[~geo["gene"].astype(str).str.match(r"^ensg\d+$", na=False)]
print(f"\ngeo_expr: {len(geo_genes)} unique gene values")
print(f"geo_expr: {len(bad_geo)} rows with non-clean ENSG format")

depmap_expr: 53961 clean ENSG columns
depmap_expr: 0 columns did NOT match clean ENSG pattern
Sample non-ENSG columns: []

hpa_rna: 20162 unique gene values
hpa_rna: 0 rows with non-clean ENSG format

geo_expr: 19914 unique gene values
geo_expr: 0 rows with non-clean ENSG format


### Gene overlap across all three

In [12]:
print(f"depmap ∩ hpa:        {len(depmap_genes & hpa_genes)}")
print(f"depmap ∩ geo:         {len(depmap_genes & geo_genes)}")
print(f"hpa ∩ geo:            {len(hpa_genes & geo_genes)}")
print(f"all three:            {len(depmap_genes & hpa_genes & geo_genes)}")
print(f"depmap only:          {len(depmap_genes - hpa_genes - geo_genes)}")
print(f"hpa only:             {len(hpa_genes - depmap_genes - geo_genes)}")
print(f"geo only:              {len(geo_genes - depmap_genes - hpa_genes)}")

depmap ∩ hpa:        19896
depmap ∩ geo:         19894
hpa ∩ geo:            16062
all three:            16060
depmap only:          30231
hpa only:             264
geo only:              18


### Cell line key resolution for depmap_expr

In [15]:
depmap_profiles = pd.read_parquet("../../data/parquet/data_clean/depmap_profiles_clean.parquet")
sample_info     = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

# depmap_expr's row index is PR- ProfileID (per clean_depmap_expr,
# the index was lowercased but not otherwise transformed)
print(depmap.index[:5].tolist())

# Check PR- IDs in depmap_expr actually exist in depmap_profiles
depmap_pr_ids = set(depmap.index)
profile_pr_ids = set(depmap_profiles["profileid"].dropna())

print(f"depmap_expr PR- IDs:    {len(depmap_pr_ids)}")
print(f"In depmap_profiles too: {len(depmap_pr_ids & profile_pr_ids)}")
print(f"Missing from bridge:    {len(depmap_pr_ids - profile_pr_ids)}")

# Filter depmap_profiles to RNA datatype only, then bridge to ACH-
rna_profiles = depmap_profiles[depmap_profiles["datatype"] == "rna"]
pr_to_ach = dict(zip(rna_profiles["profileid"], rna_profiles["modelid"]))

depmap_ach = depmap.index.map(pr_to_ach)
print(f"Resolved to ACH-: {depmap_ach.notna().sum()} / {len(depmap)}")

# Now confirm ACH- exists in sample_info — the final hop
ach_in_sample_info = set(sample_info["depmap_id"].dropna())
resolved_ach = set(depmap_ach.dropna())
print(f"ACH- found in sample_info: {len(resolved_ach & ach_in_sample_info)} / {len(resolved_ach)}")

['pr-adbjpg', 'pr-i2azwg', 'pr-5ekaac', 'pr-i21681', 'pr-i9drp1']
depmap_expr PR- IDs:    1495
In depmap_profiles too: 1495
Missing from bridge:    0
Resolved to ACH-: 1495 / 1495
ACH- found in sample_info: 1412 / 1479


In [25]:
# Your orphaned depmap_expr cell lines (failed sample_info match)
orphaned_ach_ids = set(depmap_ach.dropna()) - set(sample_info["depmap_id"])


signatures     = pd.read_parquet("../../data/parquet/data_clean/signatures_clean.parquet")
# Check if they exist in the OTHER DepMap files instead
in_depmap_profiles = orphaned_ach_ids & set(depmap_profiles["modelid"])
in_signatures = orphaned_ach_ids & set(signatures["modelid"])  # if you have access

print(f"Orphaned against sample_info: {len(orphaned_ach_ids)}")
print(f"Of those, found in depmap_profiles: {len(in_depmap_profiles)}")
print(f"Of those, found in signatures: {len(in_signatures)}")

Orphaned against sample_info: 67
Of those, found in depmap_profiles: 67
Of those, found in signatures: 42


In [26]:
remaining_25 = orphaned_ach_ids - in_signatures
print(remaining_25)

# Check if signatures itself has a default-entry filter that
# might be excluding these legitimately (recall File 14 needed
# IsDefaultEntryForModel == "Yes" filtering before use)
print(signatures[signatures["modelid"].isin(remaining_25)])
# vs checking the unfiltered raw signatures count for these IDs

{'ach-003148', 'ach-003153', 'ach-003143', 'ach-003133', 'ach-003139', 'ach-003147', 'ach-003134', 'ach-003159', 'ach-003138', 'ach-003142', 'ach-003136', 'ach-003161', 'ach-003160', 'ach-003132', 'ach-003154', 'ach-003152', 'ach-003149', 'ach-003145', 'ach-003155', 'ach-003135', 'ach-003158', 'ach-003157', 'ach-003141', 'ach-003156', 'ach-003150'}
Empty DataFrame
Columns: [signature_index, sequencingid, modelid, modelconditionid, isdefaultentryformodel, isdefaultentryformc, msiscore, lohfraction, wgd, cin, ploidy, aneuploidy]
Index: []


In [27]:
# Check the full numeric ID range present in EACH file, not just
# the unresolved subset — this tells you if depmap_profiles
# generally extends further than sample_info and signatures

import re

def get_id_numbers(id_set):
    return sorted(int(re.search(r'\d+', x).group()) for x in id_set)

profiles_ids = get_id_numbers(set(depmap_profiles["modelid"]))
sample_info_ids = get_id_numbers(set(sample_info["depmap_id"]))
signatures_ids = get_id_numbers(set(signatures["modelid"]))

print(f"depmap_profiles max ID: ACH-{profiles_ids[-1]:06d}")
print(f"sample_info max ID:     ACH-{sample_info_ids[-1]:06d}")
print(f"signatures max ID:      ACH-{signatures_ids[-1]:06d}")

depmap_profiles max ID: ACH-003161
sample_info max ID:     ACH-002926
signatures max ID:      ACH-003480


In [28]:
log_entries = []
for ach_id in orphaned_ach_ids:
    if ach_id in in_signatures:
        reason = "verified_in_signatures_missing_from_sample_info"
    else:
        reason = "unresolvable_newer_than_sample_info_no_signature_data"
    log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

unmapped_log = pd.DataFrame(log_entries)
unmapped_log.to_csv("unmapped_depmap_expr.csv", index=False)

{Ellipsis}

In [29]:
# Are the 42 contiguous too, or scattered? This tells you whether
# they're one batch or several different historical additions
print(sorted(in_signatures))

['ach-001437', 'ach-001438', 'ach-001481', 'ach-001508', 'ach-001537', 'ach-001672', 'ach-001679', 'ach-001691', 'ach-001693', 'ach-001705', 'ach-001847', 'ach-001854', 'ach-001855', 'ach-001971', 'ach-001975', 'ach-001986', 'ach-001990', 'ach-002035', 'ach-002070', 'ach-002485', 'ach-002486', 'ach-002497', 'ach-002522', 'ach-002523', 'ach-002524', 'ach-002526', 'ach-002531', 'ach-002533', 'ach-002535', 'ach-002539', 'ach-002650', 'ach-002654', 'ach-002780', 'ach-002781', 'ach-002784', 'ach-002787', 'ach-002799', 'ach-002801', 'ach-002806', 'ach-002921', 'ach-002922', 'ach-002925']


In [30]:
cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")
# Check if Cellosaurus's cross-reference column contains any of your orphaned ACH- IDs
# (Cellosaurus stores DepMap IDs as cross-references in some entries)
orphan_check = cellosaurus[cellosaurus.apply(
    lambda row: any(oid in str(row.values) for oid in orphaned_ach_ids), axis=1
)]

In [31]:
print(orphan_check.shape)
print(len(orphan_check))

(26, 17)
26


In [32]:
# Step 1 — find the column that holds DepMap ACH- references
for col in cellosaurus.columns:
    sample = cellosaurus[col].dropna().astype(str)
    if sample.str.contains("ach-", case=False, na=False).any():
        print(f"Column '{col}' contains ACH- style values")
        print(sample[sample.str.contains("ach-", case=False, na=False)].head(3).tolist())

Column 'cellosaurus_cell_line_name' contains ACH- style values
['ach-1p', 'ach-2', 'ach-3p']
Column 'synonyms' contains ACH- style values
['mz-sto-1; mainz-stomach-1', 'rat glandular stomach-2', 'rat glandular stomach-3a']
Column 'cross-references' contains ACH- style values
['cancercelllines; cvcl_n588 || cell_model_passport; sidm01315 || cosmic; 2307723 || depmap; ach-001270 || wikidata; q54581310', 'bto; bto:0003605 || clo; clo_0001072 || mccl; mcc:0000001 || cldb; cl5 || biosample; samn03471026 || cancercelllines; cvcl_0110 || cell_model_passport; sidm01967 || depmap; ach-001000 || ecacc; 86030402 || geo; gsm827421 || geo; gsm886835 || geo; gsm887898 || geo; gsm1374373 || geo; gsm3034453 || geo; gsm3034454 || geo; gsm3034455 || geo; gsm3034456 || geo; gsm3034457 || geo; gsm3034458 || iarc_tp53; 21148 || izsler; bs tcl 109 || lincs_ldp; lcl-1393 || ncbi_iran; c118 || pharmacodb; 1321n1_2_2019 || progenetix; cvcl_0110 || wikidata; q54581478', 'bto; bto:0003167 || clo; clo_0001084 || 

In [33]:
import re

def extract_depmap_id(xref_string):
    if pd.isna(xref_string):
        return None
    match = re.search(r"depmap;\s*(ach-\d+)", str(xref_string))
    return match.group(1) if match else None

cellosaurus["depmap_id_extracted"] = cellosaurus["cross-references"].apply(extract_depmap_id)

# Now do a precise, real membership check
found_in_cellosaurus = cellosaurus[
    cellosaurus["depmap_id_extracted"].isin(orphaned_ach_ids)
]
print(f"Of {len(orphaned_ach_ids)} orphaned IDs, found in Cellosaurus cross-references: {len(found_in_cellosaurus)}")
print(found_in_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession", "depmap_id_extracted"]])

Of 67 orphaned IDs, found in Cellosaurus cross-references: 26
       cellosaurus_cell_line_name cellosaurus_accession depmap_id_extracted
1020                       1618-k             cvcl_s505          ach-001437
1151                  1777n rpmet             cvcl_2277          ach-001438
35361                     chla-90             cvcl_6610          ach-001481
51868                     etcc016             cvcl_y086          ach-002650
83313                       hkbmm             cvcl_8162          ach-002070
96003                      jve015             cvcl_eg16          ach-002654
97144                    kku-m055             cvcl_m258          ach-001537
99053                       lcam1             cvcl_e062          ach-002035
102365                    maver-1             cvcl_1831          ach-002485
104231                     mes-ov             cvcl_cz92          ach-002486
105425                      mm466             cvcl_4449          ach-001971
107672                ncc-

In [34]:
def extract_discontinued_status(comment_string):
    if pd.isna(comment_string):
        return None
    match = re.search(r"discontinued:\s*depmap;\s*(ach-\d+);\s*true", str(comment_string))
    return match.group(1) if match else None

cellosaurus["discontinued_depmap_id"] = cellosaurus["comments"].apply(extract_discontinued_status)

discontinued_ids = set(cellosaurus["discontinued_depmap_id"].dropna())

overlap_with_orphans = orphaned_ach_ids & discontinued_ids
print(f"Of my {len(orphaned_ach_ids)} orphaned IDs, confirmed DISCONTINUED by DepMap: {len(overlap_with_orphans)}")
print(overlap_with_orphans)

Of my 67 orphaned IDs, confirmed DISCONTINUED by DepMap: 0
set()


In [40]:
group_25 = remaining_25  # your tight-block IDs from earlier
group_42 = orphaned_ach_ids - group_25

found_set = set(found_in_cellosaurus["depmap_id_extracted"])

print(f"Of the 26 recovered, from group_25 (tight block): {len(found_set & group_25)}")
print(f"Of the 26 recovered, from group_42 (scattered):   {len(found_set & group_42)}")

Of the 26 recovered, from group_25 (tight block): 0
Of the 26 recovered, from group_42 (scattered):   26


In [41]:
print(len(group_25)), print(len(found_set))

25
26


(None, None)

In [39]:
log_entries = []
crosswalk_additions = []

for ach_id in orphaned_ach_ids:
    if ach_id in found_set:  # the 26 recovered via Cellosaurus cross-reference
        # Pull the matching row from your earlier Cellosaurus lookup
        match = found_in_cellosaurus[
            found_in_cellosaurus["depmap_id_extracted"] == ach_id
        ].iloc[0]

        crosswalk_additions.append({
            "model_id": ach_id,
            "cvcl_id": match["cellosaurus_accession"],
            "canonical_name": match["cellosaurus_cell_line_name"],
            "resolution_method": "cellosaurus_xref_fallback"
        })

    elif ach_id in group_25:  # tight block, confirmed absent from all 3 sources
        reason = "unresolvable_too_new_confirmed_3way"
        log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

    else:  # remaining 16 from group_42, real but missing from sample_info
        reason = "verified_real_missing_from_sample_info"
        log_entries.append({"ach_id": ach_id, "dataset": "depmap_expr", "reason": reason})

# Save the 41 genuinely unresolved
unmapped_log = pd.DataFrame(log_entries)
unmapped_log.to_csv("unmapped_depmap_expr.csv", index=False)
print(f"Unmapped log: {len(unmapped_log)} rows")  # should be 41

# Save the 26 newly resolved, ready to merge into your crosswalk
crosswalk_fallback = pd.DataFrame(crosswalk_additions)
crosswalk_fallback.to_csv("crosswalk_additions_cellosaurus_fallback.csv", index=False)
print(f"Crosswalk additions: {len(crosswalk_fallback)} rows")  # should be 26

Unmapped log: 41 rows
Crosswalk additions: 26 rows


Everything so far has been investigation (figuring out match rates, finding orphans, testing fallbacks)

### Now creating final depmap_expr cell line crosswalk

In [43]:
print(len(pr_to_ach))           # PR- → ACH- mapping dict, from depmap_profiles
print(len(orphaned_ach_ids))    # the 67
print(len(found_set))           # the 26 recovered via Cellosaurus
print(len(group_25))            # the 25 too-new
sample_info[["depmap_id", "rrid", "cell_line_name"]].head()
found_in_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession", "depmap_id_extracted"]].head()

1495
67
26
25


,cellosaurus_cell_line_name,cellosaurus_accession,depmap_id_extracted
1020,1618-k,cvcl_s505,ach-001437
1151,1777n rpmet,cvcl_2277,ach-001438
35361,chla-90,cvcl_6610,ach-001481
51868,etcc016,cvcl_y086,ach-002650
83313,hkbmm,cvcl_8162,ach-002070


### Building the full PR- → ACH- mapping table, not just the orphans

In [44]:
all_profile_ids = depmap.index  # every PR- ID in your depmap_expr table

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

print(crosswalk.shape)
print(crosswalk["model_id"].isna().sum())  # PR- IDs with no ACH- at all — different problem, check separately

(1495, 2)
0


### Handle the dual-profile issue 
We flagged earlier that some cell lines have two PR- IDs pointing to the same ACH-. Resolve this now, since it affects row counts downstream:

In [45]:
dupe_models = crosswalk["model_id"].value_counts()
dupes = dupe_models[dupe_models > 1]
print(f"Cell lines with multiple RNA profiles: {len(dupes)}")

# Decision: keep highest-coverage profile per model_id
# (using total expression sum as a coverage proxy, per earlier agreement)
coverage = depmap.sum(axis=1)  # sum across genes per PR- row
crosswalk["coverage"] = crosswalk["profile_id"].map(coverage)

crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
)
print(f"After dedup: {len(crosswalk)} rows")

Cell lines with multiple RNA profiles: 16
After dedup: 1479 rows


### Attaching identity for the cleanly resolved group (matched sample_info directly)

In [53]:
resolved_mask = crosswalk["model_id"].isin(sample_info["depmap_id"])

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

crosswalk["resolution_method"] = None
crosswalk.loc[resolved_mask, "resolution_method"] = "sample_info_direct"
crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

In [47]:
print(type(resolved_mask))
print(resolved_mask.shape)

print(crosswalk.columns.tolist())
print(crosswalk.columns[crosswalk.columns.duplicated()])  # any duplicate column names?

print(type(crosswalk["model_id"]))  # should say Series — if it says DataFrame, that's the bug

<class 'pandas.core.series.Series'>
(1479,)
['profile_id', 'model_id', 'coverage', 'depmap_id', 'rrid', 'cell_line_name', 'lineage', 'primary_disease', 'resolution_method']
Index([], dtype='object')
<class 'pandas.core.series.Series'>


In [55]:
print(resolved_mask.index[:10].tolist())
print(crosswalk.index[:10].tolist())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [54]:
crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
    .reset_index(drop=True)   # ← add this, fixes the root cause generally
)

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

# Compute the mask AFTER the merge, using a column that's actually
# part of crosswalk right now — guaranteed correct index alignment
crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"

crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

print(crosswalk["resolution_method"].value_counts(dropna=False))

KeyError: 'coverage'

In [56]:
crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"

print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"Total rows: {len(crosswalk)}")  # should be 1479

resolution_method
sample_info_direct    1428
None                    67
Name: count, dtype: int64
Total rows: 1495


In [57]:
all_profile_ids = depmap.index

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

coverage = depmap.sum(axis=1)
crosswalk["coverage"] = crosswalk["profile_id"].map(coverage)

crosswalk = (
    crosswalk
    .sort_values("coverage", ascending=False)
    .drop_duplicates(subset="model_id", keep="first")
    .reset_index(drop=True)
)

print(f"After dedup: {len(crosswalk)} rows")  # should be 1479, not 1495

crosswalk = crosswalk.merge(
    sample_info[["depmap_id", "rrid", "cell_line_name", "lineage", "primary_disease"]],
    left_on="model_id", right_on="depmap_id", how="left"
)

crosswalk["resolution_method"] = None
crosswalk.loc[crosswalk["depmap_id"].notna(), "resolution_method"] = "sample_info_direct"
crosswalk = crosswalk.rename(columns={"rrid": "cvcl_id", "cell_line_name": "canonical_name"})

print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"Total rows: {len(crosswalk)}")  # should now be 1479

After dedup: 1479 rows
resolution_method
sample_info_direct    1412
None                    67
Name: count, dtype: int64
Total rows: 1479


### Attaching identity for the 26 recovered via Cellosaurus fallback

In [58]:
fallback_mask = crosswalk["model_id"].isin(found_set)

for idx in crosswalk[fallback_mask].index:
    ach_id = crosswalk.loc[idx, "model_id"]
    match = found_in_cellosaurus[found_in_cellosaurus["depmap_id_extracted"] == ach_id]
    if len(match) > 0:
        crosswalk.loc[idx, "cvcl_id"] = match.iloc[0]["cellosaurus_accession"]
        crosswalk.loc[idx, "canonical_name"] = match.iloc[0]["cellosaurus_cell_line_name"]
        crosswalk.loc[idx, "resolution_method"] = "cellosaurus_xref_fallback"

### Marking the remaining 41 as explicitly unresolved (don't drop, don't fill blanks silently)

In [59]:
unresolved_mask = crosswalk["cvcl_id"].isna()
crosswalk.loc[unresolved_mask & crosswalk["model_id"].isin(group_25), "resolution_method"] = "unresolvable_too_new_confirmed_3way"
crosswalk.loc[unresolved_mask & ~crosswalk["model_id"].isin(group_25), "resolution_method"] = "verified_real_missing_from_sample_info"

print(crosswalk["resolution_method"].value_counts())

resolution_method
sample_info_direct                        1412
cellosaurus_xref_fallback                   26
unresolvable_too_new_confirmed_3way         25
verified_real_missing_from_sample_info      16
Name: count, dtype: int64


### Validating

In [60]:
assert crosswalk["model_id"].notna().all(), "Null model_id found!"
assert crosswalk["model_id"].is_unique, "Duplicate model_id found!"
print(f"Total rows: {len(crosswalk)}")
print(f"Resolved (any method): {crosswalk['cvcl_id'].notna().sum()}")
print(f"Unresolved: {crosswalk['cvcl_id'].isna().sum()}")

Total rows: 1479
Resolved (any method): 1438
Unresolved: 41


### Saving files

In [61]:
final_crosswalk = crosswalk[crosswalk["cvcl_id"].notna()][
    ["model_id", "cvcl_id", "canonical_name", "lineage", "primary_disease", "resolution_method"]
]
unmapped = crosswalk[crosswalk["cvcl_id"].isna()][
    ["model_id", "profile_id", "resolution_method"]
]

final_crosswalk.to_csv("depmap_expr_cell_line_crosswalk.csv", index=False)
unmapped.to_csv("unmapped_depmap_expr.csv", index=False)

print(f"Final crosswalk: {len(final_crosswalk)} rows")
print(f"Unmapped log: {len(unmapped)} rows")

Final crosswalk: 1438 rows
Unmapped log: 41 rows


In [51]:
all_profile_ids = depmap.index  # every PR- ID in your depmap_expr table

crosswalk = pd.DataFrame({
    "profile_id": all_profile_ids,
    "model_id": [pr_to_ach.get(pid) for pid in all_profile_ids]
})

print(crosswalk.shape)
print(crosswalk["model_id"].isna().sum())  # PR- IDs with no ACH- at all — different problem, check separately

(1495, 2)
0


### HPA_RNA Resolution : Cell line

Resolution chain logic:
Tier 1: normalised hpa_rna name → exact match vs sample_info.cell_line_name
Tier 2: normalised hpa_rna name → match vs sample_info.stripped_cell_line_name
Tier 3: normalised hpa_rna name → look up CVCL via hpa_desc → match vs sample_info.rrid
Unresolved → log

### Running the corrected three-tier chain cleanly, end to end, and capture real numbers

In [ ]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
hpa      = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

import re
def normalise(name):
    if pd.isna(name):
        return None
    return re.sub(r"[-/_\s]", "", str(name).lower())

# Use ONE consistent column name everywhere — cln_norm
hpa["cln_norm"]         = hpa["cell line"].apply(normalise)
sample_info["cln_norm"] = sample_info["cell_line_name"].apply(normalise)
sample_info["scln_norm"] = sample_info["stripped_cell_line_name"].apply(normalise)
hpa_desc["cln_norm"]    = hpa_desc["cell line"].apply(normalise)
sample_info["rrid_norm"] = sample_info["rrid"]

tier1 = hpa["cln_norm"].isin(sample_info["cln_norm"])
print(f"Tier 1: {hpa.loc[tier1, 'cell line'].nunique()} unique cell lines")

remaining = hpa[~tier1].reset_index(drop=True)
tier2 = remaining["cln_norm"].isin(sample_info["scln_norm"])
print(f"Tier 2: {remaining.loc[tier2, 'cell line'].nunique()} additional")

still_remaining = remaining[~tier2].reset_index(drop=True)
merged = still_remaining.merge(
    hpa_desc[["cln_norm", "cellosaurus id"]], on="cln_norm", how="left"
)
tier3 = merged["cellosaurus id"].isin(sample_info["rrid_norm"])
print(f"Tier 3: {merged.loc[tier3, 'cell line'].nunique()} additional")

unresolved = merged[~tier3]
print(f"Unresolved: {unresolved['cell line'].nunique()}")

Tier 1: 1014 unique cell lines
Tier 2: 35 additional
Tier 3: 63 additional
Unresolved: 94


In [63]:
print(sorted(unresolved["cell line"].unique())[:20])
print(f"Total unresolved unique names: {unresolved['cell line'].nunique()}")

['537-mel', '624-mel', '888-mel', 'asc52telo', 'bewo', 'bj [human fibroblast]', 'bjab', 'c170', 'ccrf-sb', 'colo 206f', 'colo 320dm', 'colo 699', 'cor-l26', 'cov413b', 'deoc-1', 'fhdf/tert166', 'gtl-16', 'hacat', 'hbec3-kt', 'hbf/tert88']
Total unresolved unique names: 94


In [64]:
import re

def normalise(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)   # strip bracketed annotations
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

print(normalise("bj [human fibroblast]"))  # should now just be "bj"

bj


In [65]:
unresolved_names_norm = set(unresolved["cln_norm"].unique())  # after re-running with the fix

cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")
cellosaurus["name_norm"] = cellosaurus["cellosaurus_cell_line_name"].apply(normalise)

found_via_cellosaurus = cellosaurus[cellosaurus["name_norm"].isin(unresolved_names_norm)]
print(f"Recovered via direct Cellosaurus name match: {len(found_via_cellosaurus)}")
print(found_via_cellosaurus[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

Recovered via direct Cellosaurus name match: 93
       cellosaurus_cell_line_name cellosaurus_accession
3941                      537-mel             cvcl_8052
4254                      624-mel             cvcl_8054
4953                      888-mel             cvcl_4632
16515                   asc52telo             cvcl_u602
29300                        bewo             cvcl_0044
...                           ...                   ...
143906                   u-266/84             cvcl_0016
143908                     u-2932             cvcl_1896
144923                  ucsd-242l             cvcl_m009
145565                      uke-1             cvcl_0104
149745                  wsu-fsccl             cvcl_1903

[93 rows x 2 columns]


In [66]:
cellosaurus_syn = cellosaurus.explode("synonyms") if cellosaurus["synonyms"].dtype == "object" else cellosaurus
# adjust based on actual structure — check first:
print(cellosaurus["synonyms"].dropna().iloc[0])  # see actual format before exploding

z48-5mg-70


In [67]:
fully_resolved = set(found_via_cellosaurus["cellosaurus_cell_line_name"])
still_unresolved = unresolved_names_norm - {normalise(n) for n in fully_resolved}
print(still_unresolved)

{'bj[humanfibroblast]'}


In [69]:
recovered_cvcls = set(found_via_cellosaurus["cellosaurus_accession"])
sample_info_cvcls = set(sample_info["rrid"].dropna())

has_ach = recovered_cvcls & sample_info_cvcls
no_ach  = recovered_cvcls - sample_info_cvcls

print(f"Of 93 recovered, have a matching ACH- in sample_info: {len(has_ach)}")
print(f"Of 93 recovered, CVCL exists but NO ACH- equivalent:  {len(no_ach)}")

Of 93 recovered, have a matching ACH- in sample_info: 0
Of 93 recovered, CVCL exists but NO ACH- equivalent:  93


In [68]:
import re

def normalise(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)   # strip bracketed annotations
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

test = normalise("bj [human fibroblast]")
print(test)  # should print "bj"

# Check if "bj" resolves against sample_info directly
print(sample_info[sample_info["cln_norm"].apply(lambda x: normalise(x) if pd.notna(x) else x) == test].shape)

# Or more directly, since sample_info's cln_norm was built with the OLD
# normalise function, just check the raw cell_line_name column
print(sample_info[sample_info["cell_line_name"] == "bj"])

bj
(0, 32)
Empty DataFrame
Columns: [depmap_id, cell_line_name, stripped_cell_line_name, ccle_name, alias, cosmicid, sex, source, rrid, wtsi_master_cell_id, sample_collection_site, primary_or_metastasis, primary_disease, subtype, age, sanger_model_id, depmap_public_comments, lineage, lineage_subtype, lineage_sub_subtype, lineage_molecular_subtype, default_growth_pattern, model_manipulation, model_manipulation_details, patient_id, parent_depmap_id, cellosaurus_ncit_disease, cellosaurus_ncit_id, cellosaurus_issues, cln_norm, scln_norm, rrid_norm]
Index: []

[0 rows x 32 columns]


In [70]:
# Search loosely rather than requiring exact match
candidates = sample_info[sample_info["cell_line_name"].str.contains("bj", case=False, na=False)]
print(candidates[["cell_line_name", "stripped_cell_line_name", "alias", "rrid"]])

# Also check alias column directly, since BJ is a common enough name
# it might appear as an alias on a differently-named row
alias_candidates = sample_info[sample_info["alias"].str.contains("bj", case=False, na=False)]
print(alias_candidates[["cell_line_name", "alias", "rrid"]])

     cell_line_name stripped_cell_line_name alias       rrid
111        bj htert                 bjhtert  none  cvcl_6573
1541       va-es-bj                  vaesbj  none  cvcl_1785
Empty DataFrame
Columns: [cell_line_name, alias, rrid]
Index: []


In [71]:
test_norm = normalise("bj [human fibroblast]")  # now "bj" with the fixed function
print(test_norm)

found_bj = cellosaurus[cellosaurus["name_norm"] == test_norm]
print(found_bj[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

bj
                 cellosaurus_cell_line_name cellosaurus_accession
29746                 bj [human b-cell ihw]             cvcl_e483
29747                 bj [human fibroblast]             cvcl_3653
29748  bj [human pancreatic adenocarcinoma]             cvcl_li18


In [72]:
if len(found_bj) > 0:
    bj_cvcl = found_bj.iloc[0]["cellosaurus_accession"]
    print(f"BJ's CVCL: {bj_cvcl}")
    ach_match = sample_info[sample_info["rrid"] == bj_cvcl]
    print(ach_match[["depmap_id", "cell_line_name", "rrid"]])

BJ's CVCL: cvcl_e483
Empty DataFrame
Columns: [depmap_id, cell_line_name, rrid]
Index: []


In [73]:
def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower() if match else None

hpa_bracket_content = get_bracket_content("bj [human fibroblast]")
print(hpa_bracket_content)  # "human fibroblast"

# Now match against Cellosaurus's bracket content too, not just the base name
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

correct_match = cellosaurus[
    (cellosaurus["name_norm"] == "bj") &
    (cellosaurus["bracket_content"] == hpa_bracket_content)
]
print(correct_match[["cellosaurus_cell_line_name", "cellosaurus_accession"]])

human fibroblast
      cellosaurus_cell_line_name cellosaurus_accession
29747      bj [human fibroblast]             cvcl_3653


In [74]:
correct_cvcl = "cvcl_3653"
ach_match = sample_info[sample_info["rrid"] == correct_cvcl]
print(ach_match[["depmap_id", "cell_line_name", "rrid"]])

Empty DataFrame
Columns: [depmap_id, cell_line_name, rrid]
Index: []


In [75]:
# Check if ANY base name in Cellosaurus has multiple bracket variants
cellosaurus_bracketed = cellosaurus[cellosaurus["cellosaurus_cell_line_name"].str.contains(r"\[", na=False)]
collision_check = cellosaurus_bracketed.groupby("name_norm").size()
collisions = collision_check[collision_check > 1]
print(f"Base names with multiple bracket-variant entries in Cellosaurus: {len(collisions)}")
print(collisions)

Base names with multiple bracket-variant entries in Cellosaurus: 730
name_norm
10a6     3
10c9     4
10e2     2
10f7     2
110      2
        ..
xp1ch    2
xp1sa    3
xp1se    2
xp3hm    2
xp3le    2
Length: 730, dtype: int64


In [76]:
# Get the base names involved in any collision
collision_base_names = set(collisions.index)

# Check whether any of YOUR 93 resolved cell lines used one of these
# collision-prone base names
your_resolved_norm_names = set(found_via_cellosaurus["cellosaurus_cell_line_name"].apply(normalise))

at_risk = your_resolved_norm_names & collision_base_names
print(f"Of your 93 resolved via Cellosaurus, names with collision risk: {len(at_risk)}")
print(at_risk)

Of your 93 resolved via Cellosaurus, names with collision risk: 0
set()


In [77]:
def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower().strip() if match else None

def normalise_base(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

# Build both fields on both sides
hpa["base_norm"] = hpa["cell line"].apply(normalise_base)
hpa["bracket_content"] = hpa["cell line"].apply(get_bracket_content)

cellosaurus["base_norm"] = cellosaurus["cellosaurus_cell_line_name"].apply(normalise_base)
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

# Match on base name AND bracket content together where bracket content exists,
# fall back to base name alone only when there's no bracket on either side
def safe_match(row, cellosaurus_df):
    candidates = cellosaurus_df[cellosaurus_df["base_norm"] == row["base_norm"]]
    if len(candidates) <= 1:
        return candidates  # no ambiguity, safe to use as-is
    if row["bracket_content"] is not None:
        refined = candidates[candidates["bracket_content"] == row["bracket_content"]]
        if len(refined) >= 1:
            return refined
    return pd.DataFrame()  # genuinely ambiguous, can't safely resolve — log it

### Build the final hpa_rna cell line crosswalk

In [78]:
import re
import pandas as pd

# ── Disambiguation-aware normalisation ──────────────────────────────
def normalise_base(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_\s]", "", name)
    return name.strip()

def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower().strip() if match else None

# ── Reload clean sources fresh ──────────────────────────────────────
hpa         = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
hpa_desc    = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")
cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")

# ── Build normalised + bracket fields everywhere ────────────────────
hpa["base_norm"]        = hpa["cell line"].apply(normalise_base)
hpa["bracket_content"]  = hpa["cell line"].apply(get_bracket_content)

sample_info["cln_norm"]  = sample_info["cell_line_name"].apply(normalise_base)
sample_info["scln_norm"] = sample_info["stripped_cell_line_name"].apply(normalise_base)
sample_info["rrid_norm"] = sample_info["rrid"]

hpa_desc["cln_norm"] = hpa_desc["cell line"].apply(normalise_base)

cellosaurus["base_norm"]       = cellosaurus["cellosaurus_cell_line_name"].apply(normalise_base)
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

# ── Unique HPA cell lines to resolve ─────────────────────────────────
hpa_lines = hpa[["cell line", "base_norm", "bracket_content"]].drop_duplicates(
    subset="cell line"
).reset_index(drop=True)

results = []

for _, row in hpa_lines.iterrows():
    name, base, bracket = row["cell line"], row["base_norm"], row["bracket_content"]
    resolved = False

    # Tier 1 — exact match vs sample_info.cell_line_name
    m = sample_info[sample_info["cln_norm"] == base]
    if len(m) == 1:
        results.append({
            "hpa_name": name, "model_id": m.iloc[0]["depmap_id"],
            "cvcl_id": m.iloc[0]["rrid"], "canonical_name": m.iloc[0]["cell_line_name"],
            "resolution_method": "sample_info_exact"
        })
        continue

    # Tier 2 — stripped name match
    m = sample_info[sample_info["scln_norm"] == base]
    if len(m) == 1:
        results.append({
            "hpa_name": name, "model_id": m.iloc[0]["depmap_id"],
            "cvcl_id": m.iloc[0]["rrid"], "canonical_name": m.iloc[0]["cell_line_name"],
            "resolution_method": "sample_info_stripped"
        })
        continue

    # Tier 3 — via hpa_desc CVCL fallback
    desc_match = hpa_desc[hpa_desc["cln_norm"] == base]
    if len(desc_match) >= 1:
        cvcl = desc_match.iloc[0]["cellosaurus id"]
        m = sample_info[sample_info["rrid_norm"] == cvcl]
        if len(m) == 1:
            results.append({
                "hpa_name": name, "model_id": m.iloc[0]["depmap_id"],
                "cvcl_id": cvcl, "canonical_name": m.iloc[0]["cell_line_name"],
                "resolution_method": "hpa_desc_cvcl_fallback"
            })
            continue

    # Tier 4 — Cellosaurus name fallback, collision-aware
    candidates = cellosaurus[cellosaurus["base_norm"] == base]
    if len(candidates) == 1:
        chosen = candidates.iloc[0]
    elif len(candidates) > 1 and bracket is not None:
        refined = candidates[candidates["bracket_content"] == bracket]
        chosen = refined.iloc[0] if len(refined) == 1 else None
    else:
        chosen = None

    if chosen is not None:
        cvcl = chosen["cellosaurus_accession"]
        m = sample_info[sample_info["rrid_norm"] == cvcl]
        if len(m) == 1:
            results.append({
                "hpa_name": name, "model_id": m.iloc[0]["depmap_id"],
                "cvcl_id": cvcl, "canonical_name": chosen["cellosaurus_cell_line_name"],
                "resolution_method": "cellosaurus_name_fallback"
            })
            continue
        else:
            # CVCL found and disambiguated, but no DepMap ACH- equivalent
            results.append({
                "hpa_name": name, "model_id": None,
                "cvcl_id": cvcl, "canonical_name": chosen["cellosaurus_cell_line_name"],
                "resolution_method": "hpa_only_no_depmap_equivalent"
            })
            continue

    # Genuinely unresolved
    results.append({
        "hpa_name": name, "model_id": None, "cvcl_id": None,
        "canonical_name": None, "resolution_method": "unresolved"
    })

crosswalk = pd.DataFrame(results)

print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"Total HPA cell lines: {len(crosswalk)}")

resolution_method
sample_info_exact                1024
hpa_only_no_depmap_equivalent      97
hpa_desc_cvcl_fallback             43
sample_info_stripped               37
unresolved                          5
Name: count, dtype: int64
Total HPA cell lines: 1206


In [79]:
sample_check = crosswalk[crosswalk["resolution_method"] == "hpa_only_no_depmap_equivalent"].sample(5)
print(sample_check[["hpa_name", "canonical_name", "cvcl_id"]])

      hpa_name canonical_name    cvcl_id
546    lhcn-m2        lhcn-m2  cvcl_8890
658  nce-g 118      nce-g 118  cvcl_n741
287    hcc2911        hcc2911  cvcl_0v16
158    cov413b        cov413b  cvcl_2423
243   hbec3-kt       hbec3-kt  cvcl_x491


In [80]:
print(crosswalk[crosswalk["resolution_method"] == "unresolved"][["hpa_name"]])

                            hpa_name
45           asc2telo differentiated
68           bj htert+ sv40 large t+
69   bj htert+ sv40 large t+ rasg12v
331                           hhstec
394                            hskmc


In [81]:
assert crosswalk["hpa_name"].is_unique, "Duplicate HPA names found!"
print(f"Total resolved (model_id present): {crosswalk['model_id'].notna().sum()}")
print(f"Identified but no DepMap equivalent: {(crosswalk['resolution_method']=='hpa_only_no_depmap_equivalent').sum()}")
print(f"Genuinely unresolved: {(crosswalk['resolution_method']=='unresolved').sum()}")

Total resolved (model_id present): 1104
Identified but no DepMap equivalent: 97
Genuinely unresolved: 5


In [82]:
# Quick verification this isn't a remaining normalisation issue —
# check if removing the "+" and engineering descriptors gets anywhere
test_names = ["bj htert+ sv40 large t+", "bj htert+ sv40 large t+ rasg12v"]
for name in test_names:
    print(name, "→", normalise_base(name))

bj htert+ sv40 large t+ → bjhtert+sv40larget+
bj htert+ sv40 large t+ rasg12v → bjhtert+sv40larget+rasg12v


In [83]:
# Fix the normalise function to also strip '+'
def normalise_base(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_+\s]", "", name)   # added + here
    return name.strip()

# Re-test
for name in ["bj htert+ sv40 large t+", "bj htert+ sv40 large t+ rasg12v"]:
    print(name, "→", normalise_base(name))

bj htert+ sv40 large t+ → bjhtertsv40larget
bj htert+ sv40 large t+ rasg12v → bjhtertsv40largetrasg12v


In [84]:
test_norm_1 = normalise_base("bj htert+ sv40 large t+")
test_norm_2 = normalise_base("bj htert+ sv40 large t+ rasg12v")

print(cellosaurus[cellosaurus["base_norm"] == test_norm_1][["cellosaurus_cell_line_name", "cellosaurus_accession"]])
print(cellosaurus[cellosaurus["base_norm"] == test_norm_2][["cellosaurus_cell_line_name", "cellosaurus_accession"]])

print(sample_info[sample_info["cln_norm"] == test_norm_1][["cell_line_name", "rrid"]])
print(sample_info[sample_info["cln_norm"] == test_norm_2][["cell_line_name", "rrid"]])

Empty DataFrame
Columns: [cellosaurus_cell_line_name, cellosaurus_accession]
Index: []
Empty DataFrame
Columns: [cellosaurus_cell_line_name, cellosaurus_accession]
Index: []
Empty DataFrame
Columns: [cell_line_name, rrid]
Index: []
Empty DataFrame
Columns: [cell_line_name, rrid]
Index: []


In [85]:
final_crosswalk = crosswalk[crosswalk["model_id"].notna()][
    ["hpa_name", "model_id", "cvcl_id", "canonical_name", "resolution_method"]
]
identified_no_depmap = crosswalk[crosswalk["resolution_method"] == "hpa_only_no_depmap_equivalent"]
unresolved = crosswalk[crosswalk["resolution_method"] == "unresolved"]

final_crosswalk.to_csv("hpa_rna_cell_line_crosswalk.csv", index=False)
identified_no_depmap.to_csv("hpa_rna_no_depmap_equivalent.csv", index=False)
unresolved.to_csv("unmapped_hpa_rna.csv", index=False)

print(f"Final crosswalk (has model_id): {len(final_crosswalk)}")
print(f"Identified, no DepMap (separate file): {len(identified_no_depmap)}")
print(f"Unmapped: {len(unresolved)}")

Final crosswalk (has model_id): 1104
Identified, no DepMap (separate file): 97
Unmapped: 5


In [86]:
# Get the CVCLs for your 97 "no depmap equivalent" rows
no_depmap_cvcls = set(
    crosswalk[crosswalk["resolution_method"] == "hpa_only_no_depmap_equivalent"]["cvcl_id"]
)

# Check if any of these CVCLs actually DO appear somewhere else —
# e.g. in depmap_profiles via a Cellosaurus cross-reference to an
# ACH- ID that simply isn't in THIS sample_info snapshot

# First, does the Cellosaurus cross-references field have a DepMap
# entry for any of these, even if sample_info doesn't?
no_depmap_names = crosswalk[
    crosswalk["resolution_method"] == "hpa_only_no_depmap_equivalent"
]["canonical_name"].tolist()

cellosaurus_rows = cellosaurus[
    cellosaurus["cellosaurus_cell_line_name"].isin(no_depmap_names)
]

import re
def extract_depmap_id(xref_string):
    if pd.isna(xref_string):
        return None
    match = re.search(r"depmap;\s*(ach-\d+)", str(xref_string))
    return match.group(1) if match else None

cellosaurus_rows["depmap_id_extracted"] = cellosaurus_rows["cross-references"].apply(extract_depmap_id)

has_depmap_xref = cellosaurus_rows[cellosaurus_rows["depmap_id_extracted"].notna()]
print(f"Of 97 'no DepMap equivalent', Cellosaurus actually shows a DepMap cross-ref for: {len(has_depmap_xref)}")
print(has_depmap_xref[["cellosaurus_cell_line_name", "depmap_id_extracted"]])

Of 97 'no DepMap equivalent', Cellosaurus actually shows a DepMap cross-ref for: 4
       cellosaurus_cell_line_name depmap_id_extracted
37356                    colo 699          ach-001041
108073                  nci-h2077          ach-000010
108279                   nci-h820          ach-001140
149745                  wsu-fsccl          ach-001708


/var/folders/w4/fbz2zhr165g6wr5k09mtgl9r0000gn/T/ipykernel_20920/1355445331.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cellosaurus_rows["depmap_id_extracted"] = cellosaurus_rows["cross-references"].apply(extract_depmap_id)


In [20]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
print(hpa_desc.columns.tolist())

['cell line', 'disease', 'disease subtype', 'cellosaurus id', 'patient', 'primary/metastasis', 'sample collection site']


In [21]:
# Search for likely candidates rather than assuming
for col in hpa_desc.columns:
    if "cell" in col or "cvcl" in col.lower() or "cellosaurus" in col.lower():
        print(col, "→ sample:", hpa_desc[col].dropna().iloc[:3].tolist())

cell line → sample: ['143b', '22rv1', '23132/87']
cellosaurus id → sample: ['cvcl_2270', 'cvcl_1045', 'cvcl_1046']


In [22]:
test_name = "23132/87"
test_norm = normalise(test_name)
print(f"'{test_name}' normalises to: '{test_norm}'")

# Does it match anything in sample_info?
match = sample_info[sample_info["cln_norm"] == test_norm]  # or whichever norm col you used
print(match)

# Does it match anything in hpa_desc?
match2 = hpa_desc[hpa_desc["cln_norm"] == test_norm]
print(match2)

'23132/87' normalises to: '2313287'
       depmap_id cell_line_name stripped_cell_line_name        ccle_name  \
1241  ach-000948       23132/87                 2313287  2313287_stomach   

     alias  cosmicid   sex source       rrid  wtsi_master_cell_id  ...  \
1241  none  910924.0  male   dsmz  cvcl_1046                558.0  ...   

     model_manipulation model_manipulation_details patient_id  \
1241               none                       none  pt-vdirwk   

     parent_depmap_id cellosaurus_ncit_disease cellosaurus_ncit_id  \
1241             none   gastric adenocarcinoma               c4004   

     cellosaurus_issues cln_norm scln_norm  rrid_norm  
1241               none  2313287   2313287  cvcl_1046  

[1 rows x 32 columns]


KeyError: 'cln_norm'

### Cell line key resolution for hpa_rna, using hpa_desc

In [23]:
hpa_desc = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")

def normalise(name):
    if pd.isna(name):
        return None
    return re.sub(r"[-/_\s]", "", str(name).lower())

hpa["cell_line_norm"] = hpa["cell line"].apply(normalise)
sample_info["cln_norm"]  = sample_info["cell_line_name"].apply(normalise)
sample_info["scln_norm"] = sample_info["stripped_cell_line_name"].apply(normalise)

# Tier 1 — direct name match
tier1 = hpa["cell_line_norm"].isin(sample_info["cln_norm"])
print(f"Tier 1 (direct name): {hpa.loc[tier1, 'cell line'].nunique()} unique cell lines")

# Tier 2 — stripped name match, only for what tier 1 missed
remaining = hpa[~tier1]
tier2 = remaining["cell_line_norm"].isin(sample_info["scln_norm"])
print(f"Tier 2 (stripped name): {remaining.loc[tier2, 'cell line'].nunique()} additional")

# Tier 3 — go through hpa_desc to get CVCL, then match sample_info.rrid
still_remaining = remaining[~tier2]
hpa_desc["cln_norm"] = hpa_desc["cell line"].apply(normalise)  # confirm actual col name
sample_info["rrid_norm"] = sample_info["rrid"]  # already lowercase cvcl_xxxx format

merged = still_remaining.merge(
    hpa_desc[["cln_norm", "cellosaurus id"]],  # confirm actual col name from your data
    on="cln_norm", how="left"
)
tier3 = merged["cellosaurus id"].isin(sample_info["rrid_norm"])
print(f"Tier 3 (via hpa_desc CVCL): {merged.loc[tier3, 'cell line'].nunique()} additional")

unresolved = merged[~tier3]
print(f"Permanently unresolved: {unresolved['cell line'].nunique()}")

Tier 1 (direct name): 1014 unique cell lines
Tier 2 (stripped name): 35 additional


KeyError: 'cln_norm'

In [18]:
sample_info.head(5)

,depmap_id,cell_line_name,stripped_cell_line_name,ccle_name,alias,cosmicid,sex,source,rrid,wtsi_master_cell_id,...,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,cellosaurus_ncit_disease,cellosaurus_ncit_id,cellosaurus_issues,cln_norm,scln_norm,rrid_norm
0,ach-000016,slr 21,slr21,slr21_kidney,none,NaN,none,academic lab,cvcl_v607,NaN,...,none,none,pt-jnarlb,none,clear cell renal cell carcinoma,c4033,none,slr21,slr21,cvcl_v607
1,ach-000032,mhh-call-3,mhhcall3,mhhcall3_haematopoietic_and_lymphoid_tissue,none,NaN,female,dsmz,cvcl_0089,NaN,...,none,none,pt-p2koyi,none,childhood b acute lymphoblastic leukemia,c9140,none,mhhcall3,mhhcall3,cvcl_0089
2,ach-000033,nci-h1819,ncih1819,ncih1819_lung,none,NaN,female,academic lab,cvcl_1497,NaN,...,none,none,pt-9p1wqv,none,lung adenocarcinoma,c3512,none,ncih1819,ncih1819,cvcl_1497
3,ach-000043,hs 895.t,hs895t,hs895t_fibroblast,none,NaN,female,atcc,cvcl_0993,NaN,...,none,none,pt-rtuvzq,none,melanoma,c3224,none,hs895.t,hs895t,cvcl_0993
4,ach-000049,hek te,hekte,hekte_kidney,none,NaN,none,academic lab,cvcl_ws59,NaN,...,immortalized,none,pt-qwyygr,none,none,none,no information is available about this cell li...,hekte,hekte,cvcl_ws59


In [87]:
import pandas as pd

# Load the cleaned miRNA parquet (already processed by clean_mirna())
mirna = pd.read_parquet("../../data/parquet/data_clean/mirna_clean.parquet")

print(mirna.shape)
print(mirna.columns.tolist())
print(mirna[["name", "description"]].head())

(734, 956)
['name', 'description', 'dms53_lung', 'sw1116_large_intestine', 'ncih1694_lung', 'p3hr1_haematopoietic_and_lymphoid_tissue', 'hut78_haematopoietic_and_lymphoid_tissue', 'umuc3_urinary_tract', 'hos_bone', 'huns1_haematopoietic_and_lymphoid_tissue', 'aml193_haematopoietic_and_lymphoid_tissue', 'rvh421_skin', 'ncih1184_lung', 'hcc2157_breast', 'tc71_bone', 'ncih2227_lung', 'snu449_liver', 'ncih28_pleura', 'ov56_ovary', 'jhos4_ovary', 'kyse450_oesophagus', 'rmugs_ovary', 'kle_endometrium', 'hs895t_fibroblast', 'ln229_central_nervous_system', 'p31fuj_haematopoietic_and_lymphoid_tissue', 'rkn_soft_tissue', 'patu8988s_pancreas', 'nh6_autonomic_ganglia', 'sf126_central_nervous_system', 'rerflcad2_lung', 'oums23_large_intestine', 'sngm_endometrium', 'oums27_bone', 'ncih2347_lung', 'sw1990_pancreas', 'hs940t_fibroblast', 'hs611t_haematopoietic_and_lymphoid_tissue', 'toledo_haematopoietic_and_lymphoid_tissue', 'rerfgc1b_stomach', 'ht1080_soft_tissue', 'ncih2087_lung', 'cov318_ovary', '

In [88]:
# Total rows vs unique values in description
print(f"Total rows: {len(mirna)}")
print(f"Unique description values: {mirna['description'].nunique()}")
print(f"Null description values: {mirna['description'].isna().sum()}")

# Find the actual duplicated entries
dupe_mask = mirna["description"].duplicated(keep=False)
dupes = mirna[dupe_mask].sort_values("description")

print(f"\nRows involved in a duplicate description: {len(dupes)}")
print(dupes[["name", "description"]])

Total rows: 734
Unique description values: 734
Null description values: 0

Rows involved in a duplicate description: 0
Empty DataFrame
Columns: [name, description]
Index: []


In [94]:
mirna["description"].head(10).tolist()

['hsa-let-7a',
 'hsa-let-7b',
 'hsa-let-7c',
 'hsa-let-7d',
 'hsa-let-7e',
 'hsa-let-7f',
 'hsa-let-7g',
 'hsa-let-7i',
 'hsa-mir-1',
 'hsa-mir-100']

In [91]:
depmap_profiles["profileid"].count()

3830

In [92]:
mirna["description"].isin(depmap_profiles["profileid"]).sum()

0

In [95]:
mirna = pd.read_parquet("../../data/parquet/data_clean/mirna_clean.parquet")

print(mirna.columns.tolist()[:10])
print(mirna.columns.tolist()[-10:])

['name', 'description', 'dms53_lung', 'sw1116_large_intestine', 'ncih1694_lung', 'p3hr1_haematopoietic_and_lymphoid_tissue', 'hut78_haematopoietic_and_lymphoid_tissue', 'umuc3_urinary_tract', 'hos_bone', 'huns1_haematopoietic_and_lymphoid_tissue']
['molt3_haematopoietic_and_lymphoid_tissue', 'hop62_lung', 'ekvx_lung', 'ovcar5_ovary', 'uo31_kidney', 'sf268_central_nervous_system', 'sf539_central_nervous_system', 'snb75_central_nervous_system', 'hop92_lung', 'mutz3_haematopoietic_and_lymphoid_tissue']


In [96]:
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

cell_line_cols = [c for c in mirna.columns if c not in ["name", "description"]]
print(f"Cell line columns in mirna: {len(cell_line_cols)}")

# Direct match against sample_info's CCLE_Name
matched = set(cell_line_cols) & set(sample_info["ccle_name"])
print(f"Direct CCLE_Name match: {len(matched)}")

unmatched = set(cell_line_cols) - matched
print(f"Unmatched: {len(unmatched)}")
print(list(unmatched)[:10])

Cell line columns in mirna: 954
Direct CCLE_Name match: 950
Unmatched: 4
['ke97_haematopoietic_and_lymphoid_tissue', 'ncih684_liver', 'colo699_lung', 'ncih1339_lung']


In [97]:
def strip_tissue_suffix(name):
    # CCLE format is CELLNAME_TISSUE — take everything before the first underscore
    return str(name).split("_")[0].lower()

unmatched_stripped = {strip_tissue_suffix(n): n for n in unmatched}
sample_info["scln_lower"] = sample_info["stripped_cell_line_name"].str.lower()

fallback_matched = set(unmatched_stripped.keys()) & set(sample_info["scln_lower"])
print(f"Recovered via stripped-name fallback: {len(fallback_matched)}")

still_unmatched = set(unmatched_stripped.keys()) - fallback_matched
print(f"Still unmatched: {len(still_unmatched)}")

Recovered via stripped-name fallback: 2
Still unmatched: 2


### Geo_Expr cell line resolution

In [98]:
geo_expr = pd.read_parquet("../../data/parquet/data_clean/geo_expr_clean.parquet")

In [100]:
geo_expr.shape
geo_expr.head(5)

,gene,gsm101610,gsm101615,gsm101616,gsm101667,gsm101668,gsm101671,gsm101672,gsm101673,gsm101674,...,gsm960289,gsm960290,gsm960291,gsm960292,gsm960293,gsm960294,gsm960295,gsm960296,gsm960297,gsm960298
0,ensg00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ensg00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ensg00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952
3,ensg00000000457,58.934814,95.068222,94.900459,51.810070,52.327530,50.011375,53.793667,51.172344,52.677490,...,35.778347,57.736485,25.528204,24.593216,23.337006,47.793488,41.268002,64.042831,37.363773,61.027988
4,ensg00000000460,136.418900,257.929169,271.317230,162.090073,160.913849,206.788635,209.966019,134.272583,147.408554,...,14.693585,19.083935,114.056015,137.948593,149.641708,111.793892,98.536613,167.151016,12.268769,24.296593


In [102]:
geo_info = pd.read_parquet("../../data/parquet/data_clean/geo_info_clean.parquet")

null_cvcl_gsms = set(geo_info[geo_info["cellosaurus_id"].isna()]["geo_accession"])
cvcl_7082_gsms = set(geo_info[geo_info["cellosaurus_id"] == "cvcl_7082"]["geo_accession"])

print(f"Null CVCL GSMs to exclude: {len(null_cvcl_gsms)}")    # ~108
print(f"CVCL_7082 anomaly GSMs to exclude: {len(cvcl_7082_gsms)}")  # ~478

Null CVCL GSMs to exclude: 0
CVCL_7082 anomaly GSMs to exclude: 478


In [104]:
geo = geo_expr.copy()
gsm_cols = [c for c in geo.columns if c != "gene"]

bad_gsms = null_cvcl_gsms | cvcl_7082_gsms
usable_gsm_cols = [c for c in gsm_cols if c not in bad_gsms]

print(f"Total GSM columns: {len(gsm_cols)}")
print(f"Usable after exclusions: {len(usable_gsm_cols)}")

Total GSM columns: 3267
Usable after exclusions: 2789


In [105]:
matched_via_cvcl = geo_info[
    geo_info["geo_accession"].isin(usable_gsm_cols) &
    geo_info["cellosaurus_id"].notna()
]
print(f"Resolved via cellosaurus_id directly: {len(matched_via_cvcl)}")

needs_name_fallback = geo_info[
    geo_info["geo_accession"].isin(usable_gsm_cols) &
    geo_info["cellosaurus_id"].isna()
]
print(f"Need name-column fallback: {len(needs_name_fallback)}")

Resolved via cellosaurus_id directly: 2789
Need name-column fallback: 0


In [106]:
# First, confirm geo_info is the file you think it is
print(geo_info.shape)
print(geo_info["cellosaurus_id"].isna().sum())

# Check for non-standard null representations — recall clean_str_cols()
# replaces the STRING "nan" with pd.NA, but check this actually
# applied correctly here
print(geo_info["cellosaurus_id"].unique()[:20])

# Specifically check for the string "none" or "nan" surviving as text
# rather than being a true null
suspicious = geo_info[geo_info["cellosaurus_id"].astype(str).str.lower().isin(
    ["none", "nan", "<na>", ""]
)]
print(f"Rows with suspicious null-like string values: {len(suspicious)}")
print(suspicious["cellosaurus_id"].value_counts())

(3267, 23)
0
['cvcl_0131' 'cvcl_0393' 'cvcl_1715' 'cvcl_0633' 'cvcl_0020' 'cvcl_0021'
 'cvcl_0022' 'cvcl_1051' 'cvcl_0023' 'cvcl_0025' 'cvcl_0105' 'cvcl_0038'
 'cvcl_0291' 'cvcl_0030' 'cvcl_0839' 'cvcl_0320' 'cvcl_5310' 'cvcl_0598'
 'cvcl_0031' 'cvcl_0062']
Rows with suspicious null-like string values: 108
cellosaurus_id
none    108
Name: count, dtype: int64


In [107]:
# Treat any of these as effectively null before proceeding
true_null_mask = (
    geo_info["cellosaurus_id"].isna() |
    geo_info["cellosaurus_id"].astype(str).str.lower().isin(["none", "nan", "<na>", ""])
)

null_cvcl_gsms = set(geo_info[true_null_mask]["geo_accession"])
print(f"Corrected null CVCL GSMs to exclude: {len(null_cvcl_gsms)}")

Corrected null CVCL GSMs to exclude: 108


In [108]:
true_null_mask = (
    geo_info["cellosaurus_id"].isna() |
    (geo_info["cellosaurus_id"].astype(str).str.lower() == "none")
)

null_cvcl_gsms = set(geo_info[true_null_mask]["geo_accession"])
cvcl_7082_gsms = set(geo_info[geo_info["cellosaurus_id"] == "cvcl_7082"]["geo_accession"])

print(f"Null CVCL GSMs to exclude: {len(null_cvcl_gsms)}")     # should now be 108
print(f"CVCL_7082 GSMs to exclude: {len(cvcl_7082_gsms)}")     # 478

bad_gsms = null_cvcl_gsms | cvcl_7082_gsms

gsm_cols = [c for c in geo.columns if c != "gene"]
usable_gsm_cols = [c for c in gsm_cols if c not in bad_gsms]

print(f"Total GSM columns: {len(gsm_cols)}")
print(f"Usable after corrected exclusions: {len(usable_gsm_cols)}")

Null CVCL GSMs to exclude: 108
CVCL_7082 GSMs to exclude: 478
Total GSM columns: 3267
Usable after corrected exclusions: 2681


In [109]:
matched_via_cvcl = geo_info[
    geo_info["geo_accession"].isin(usable_gsm_cols) &
    ~true_null_mask &
    (geo_info["cellosaurus_id"] != "cvcl_7082")
]
print(f"Resolved via cellosaurus_id directly: {len(matched_via_cvcl)}")

Resolved via cellosaurus_id directly: 2681


In [110]:
import pandas as pd

# ── Load sources fresh ───────────────────────────────────────────────
geo         = pd.read_parquet("../../data/parquet/data_clean/geo_expr_clean.parquet")
geo_info    = pd.read_parquet("../../data/parquet/data_clean/geo_info_clean.parquet")
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

sample_info["rrid_norm"] = sample_info["rrid"]

# ── Step 0 — corrected exclusions ────────────────────────────────────
true_null_mask = (
    geo_info["cellosaurus_id"].isna() |
    (geo_info["cellosaurus_id"].astype(str).str.lower() == "none")
)
null_cvcl_gsms = set(geo_info[true_null_mask]["geo_accession"])
cvcl_7082_gsms = set(geo_info[geo_info["cellosaurus_id"] == "cvcl_7082"]["geo_accession"])
bad_gsms = null_cvcl_gsms | cvcl_7082_gsms

print(f"Null CVCL GSMs excluded: {len(null_cvcl_gsms)}")
print(f"cvcl_7082 GSMs excluded: {len(cvcl_7082_gsms)}")
print(f"Total excluded: {len(bad_gsms)}")

gsm_cols = [c for c in geo.columns if c != "gene"]
usable_gsm_cols = [c for c in gsm_cols if c not in bad_gsms]
print(f"Total GSM columns: {len(gsm_cols)}")
print(f"Usable after exclusions: {len(usable_gsm_cols)}")

# ── Step 1 & 2 — resolve each usable GSM, GSM → CVCL → model_id ─────
results = []

for gsm in usable_gsm_cols:
    row = geo_info[geo_info["geo_accession"] == gsm]

    if len(row) == 0:
        results.append({
            "gsm": gsm, "model_id": None, "cvcl_id": None,
            "canonical_name": None, "gse_id": None,
            "resolution_method": "no_geo_info_entry"
        })
        continue

    cvcl = row.iloc[0]["cellosaurus_id"]
    gse_id = row.iloc[0]["gse_id"]
    matching_type = row.iloc[0]["matching_type"]

    sample_match = sample_info[sample_info["rrid_norm"] == cvcl]

    if len(sample_match) == 1:
        results.append({
            "gsm": gsm,
            "model_id": sample_match.iloc[0]["depmap_id"],
            "cvcl_id": cvcl,
            "canonical_name": sample_match.iloc[0]["cell_line_name"],
            "gse_id": gse_id,
            "matching_type": matching_type,
            "resolution_method": "cellosaurus_id_direct"
        })
    elif len(sample_match) > 1:
        # Multiple sample_info rows share this RRID — flag, don't guess
        results.append({
            "gsm": gsm, "model_id": None, "cvcl_id": cvcl,
            "canonical_name": None, "gse_id": gse_id,
            "matching_type": matching_type,
            "resolution_method": "ambiguous_multiple_sample_info_matches"
        })
    else:
        # CVCL is real and verified (matching_type present),
        # but no DepMap model exists for it
        results.append({
            "gsm": gsm, "model_id": None, "cvcl_id": cvcl,
            "canonical_name": None, "gse_id": gse_id,
            "matching_type": matching_type,
            "resolution_method": "geo_only_no_depmap_equivalent"
        })

geo_crosswalk = pd.DataFrame(results)

print("\n--- Resolution breakdown ---")
print(geo_crosswalk["resolution_method"].value_counts(dropna=False))
print(f"\nTotal usable GSMs processed: {len(geo_crosswalk)}")

# ── Validation ────────────────────────────────────────────────────────
assert geo_crosswalk["gsm"].is_unique, "Duplicate GSM rows found!"

resolved = geo_crosswalk["model_id"].notna().sum()
print(f"\nResolved to model_id: {resolved}")
print(f"Unique cell lines resolved: {geo_crosswalk['model_id'].nunique()}")

# ── Multi-study check, needed for the upcoming z-score transform ────
multi_study = (
    geo_crosswalk[geo_crosswalk["model_id"].notna()]
    .groupby("model_id")["gse_id"]
    .nunique()
)
print(f"\nCell lines appearing in more than one GSE study: {(multi_study > 1).sum()}")
print(f"Cell lines in exactly one study: {(multi_study == 1).sum()}")

# ── Split into final deliverables ────────────────────────────────────
final_crosswalk = geo_crosswalk[geo_crosswalk["model_id"].notna()][
    ["gsm", "model_id", "cvcl_id", "canonical_name", "gse_id", "resolution_method"]
]

no_depmap_equivalent = geo_crosswalk[
    geo_crosswalk["resolution_method"] == "geo_only_no_depmap_equivalent"
][["gsm", "cvcl_id", "gse_id", "matching_type"]]

unmapped = geo_crosswalk[
    geo_crosswalk["resolution_method"].isin(
        ["no_geo_info_entry", "ambiguous_multiple_sample_info_matches"]
    )
]

excluded_log = pd.DataFrame([
    {"gsm": g, "reason": "null_cvcl"} for g in null_cvcl_gsms
] + [
    {"gsm": g, "reason": "cvcl_7082_unverified_assignment"} for g in cvcl_7082_gsms
])

final_crosswalk.to_csv("geo_expr_cell_line_crosswalk.csv", index=False)
no_depmap_equivalent.to_csv("geo_expr_no_depmap_equivalent.csv", index=False)
unmapped.to_csv("unmapped_geo_expr.csv", index=False)
excluded_log.to_csv("excluded_geo_expr.csv", index=False)

print(f"\nFinal crosswalk: {len(final_crosswalk)} rows")
print(f"No DepMap equivalent: {len(no_depmap_equivalent)} rows")
print(f"Unmapped (genuinely unresolved): {len(unmapped)} rows")
print(f"Excluded (pre-resolution): {len(excluded_log)} rows")
print(f"Total accounted for: {len(final_crosswalk) + len(no_depmap_equivalent) + len(unmapped) + len(excluded_log)} / 3267")

Null CVCL GSMs excluded: 108
cvcl_7082 GSMs excluded: 478
Total excluded: 586
Total GSM columns: 3267
Usable after exclusions: 2681

--- Resolution breakdown ---
resolution_method
cellosaurus_id_direct                     2306
geo_only_no_depmap_equivalent              373
ambiguous_multiple_sample_info_matches       2
Name: count, dtype: int64

Total usable GSMs processed: 2681

Resolved to model_id: 2306
Unique cell lines resolved: 586

Cell lines appearing in more than one GSE study: 276
Cell lines in exactly one study: 310

Final crosswalk: 2306 rows
No DepMap equivalent: 373 rows
Unmapped (genuinely unresolved): 2 rows
Excluded (pre-resolution): 586 rows
Total accounted for: 3267 / 3267


In [111]:
sample_check = geo_crosswalk[
    geo_crosswalk["resolution_method"] == "geo_only_no_depmap_equivalent"
].sample(5)
print(sample_check[["gsm", "cvcl_id", "gse_id", "matching_type"]])

             gsm    cvcl_id    gse_id            matching_type
2378   gsm748846  cvcl_0561  gse30240     cello cell line name
2264   gsm587164  cvcl_n742  gse23806  cello cell line synonym
1444   gsm170823  cvcl_u802   gse7127     cello cell line name
1459   gsm170866  cvcl_c837   gse7127     cello cell line name
761   gsm1374374  cvcl_6344  gse57083  cello cell line synonym


In [112]:
ambiguous = geo_crosswalk[
    geo_crosswalk["resolution_method"] == "ambiguous_multiple_sample_info_matches"
]
print(ambiguous)

# Check what's actually duplicated in sample_info for these CVCLs
for cvcl in ambiguous["cvcl_id"]:
    print(f"\n--- {cvcl} ---")
    print(sample_info[sample_info["rrid_norm"] == cvcl][
        ["depmap_id", "cell_line_name", "rrid"]
    ])

             gsm model_id    cvcl_id canonical_name    gse_id  matching_type  \
843   gsm1374458     None  cvcl_1150           None  gse57083  cello geo gsm   
1600   gsm219757     None  cvcl_0041           None      none  cello geo gsm   

                           resolution_method  
843   ambiguous_multiple_sample_info_matches  
1600  ambiguous_multiple_sample_info_matches  

--- cvcl_1150 ---
       depmap_id cell_line_name       rrid
1550  ach-001737       ctv-1-dm  cvcl_1150
1711  ach-002222          ctv-1  cvcl_1150

--- cvcl_0041 ---
       depmap_id cell_line_name       rrid
1135  ach-000833          rh-30  cvcl_0041
1361  ach-001189           none  cvcl_0041


In [113]:
# Check whether the GSE study or any other context in geo_info
# gives a hint about which CTV-1 variant this specific GSM actually is
print(geo_info[geo_info["geo_accession"] == "gsm1374458"])

# For cvcl_0041, just pick the row with a real name — not "none"
correct_rh30 = sample_info[
    (sample_info["rrid_norm"] == "cvcl_0041") &
    (sample_info["cell_line_name"] != "none")
]
print(correct_rh30)

    geo_accession cel_file_names            title                 status  \
904    gsm1374458     gsm1374458  ctv-1 ghp221105  public on apr 28 2014   

    submission_date last_update_date type  channel_count     source_name_ch1  \
904     apr 25 2014      apr 28 2014  rna            1.0  az gene expression   

     organism_ch1  ... contact_institute    gse_id  \
904  homo sapiens  ...       astrazeneca  gse57083   

                      gse_filename cell_line                       disease  \
904  gse57083_series_matrix.txt.gz      ctv1  aml (acute myeloid leukemia)   

                   origin cellosaurus_id cellline  matching_type  \
904  hematopoietic system      cvcl_1150    ctv-1  cello geo gsm   

    cell_line_trimmed  
904              ctv1  

[1 rows x 23 columns]
       depmap_id cell_line_name stripped_cell_line_name         ccle_name  \
1135  ach-000833          rh-30                    rh30  rh30_soft_tissue   

     alias  cosmicid   sex source       rrid  wtsi_master

In [114]:
correct_match = sample_info[
    (sample_info["rrid_norm"] == "cvcl_1150") &
    (sample_info["cell_line_name"] == "ctv-1")
]
print(correct_match[["depmap_id", "cell_line_name"]])

       depmap_id cell_line_name
1711  ach-002222          ctv-1


In [115]:
print(sample_info[sample_info["depmap_id"] == "ach-001189"][
    ["depmap_id", "cell_line_name", "cellosaurus_issues", "rrid"]
])

       depmap_id cell_line_name                         cellosaurus_issues  \
1361  ach-001189           none  removed from depmap; removed from depmap;   

           rrid  
1361  cvcl_0041  


In [116]:
# Check how many rows in sample_info are flagged as removed/deprecated
removed_count = sample_info["cellosaurus_issues"].str.contains(
    "removed from depmap", case=False, na=False
).sum()
print(f"sample_info rows flagged as removed from DepMap: {removed_count}")

sample_info rows flagged as removed from DepMap: 11


In [117]:
def resolve_with_tiebreak(cvcl, geo_title=None, geo_cellline=None):
    matches = sample_info[sample_info["rrid_norm"] == cvcl]
    if len(matches) == 1:
        return matches.iloc[0]

    # Exclude anything flagged as removed/deprecated FIRST
    active = matches[~matches["cellosaurus_issues"].str.contains(
        "removed from depmap", case=False, na=False
    )]
    if len(active) == 1:
        return active.iloc[0]

    # If still ambiguous, try matching against GEO's own title/cellline text
    if geo_title is not None and len(active) > 1:
        title_match = active[active["cell_line_name"].apply(
            lambda x: str(x) in str(geo_title)
        )]
        if len(title_match) == 1:
            return title_match.iloc[0]

    return None  # still genuinely ambiguous

In [118]:
print(sample_info[sample_info["rrid_norm"] == "cvcl_0041"][
    ["depmap_id", "cell_line_name", "cellosaurus_issues"]
])
# Confirms: every row for this CVCL is deprecated, none usable

       depmap_id cell_line_name                         cellosaurus_issues
1135  ach-000833          rh-30  removed from depmap; removed from depmap;
1361  ach-001189           none  removed from depmap; removed from depmap;


In [119]:
def resolve_with_tiebreak(cvcl, geo_title=None):
    matches = sample_info[sample_info["rrid_norm"] == cvcl]

    if len(matches) == 0:
        return None, "no_match"

    if len(matches) == 1:
        row = matches.iloc[0]
        if "removed from depmap" in str(row["cellosaurus_issues"]).lower():
            return None, "cvcl_exists_but_only_deprecated_ach_ids"
        return row, "resolved"

    # Multiple candidates — exclude deprecated ones first
    active = matches[~matches["cellosaurus_issues"].astype(str).str.contains(
        "removed from depmap", case=False, na=False
    )]

    if len(active) == 1:
        return active.iloc[0], "resolved_after_deprecation_filter"

    if len(active) == 0:
        # ALL candidates deprecated — this is cvcl_0041's actual situation
        return None, "cvcl_exists_but_only_deprecated_ach_ids"

    # Still more than one active candidate — try matching geo_info's title
    if geo_title is not None:
        title_match = active[active["cell_line_name"].apply(
            lambda x: str(x) in str(geo_title).lower()
        )]
        if len(title_match) == 1:
            return title_match.iloc[0], "resolved_via_geo_title"

    return None, "ambiguous_unresolved"


# ── Now rebuild the crosswalk loop, calling this function ───────────
results = []

for gsm in usable_gsm_cols:
    row = geo_info[geo_info["geo_accession"] == gsm]

    if len(row) == 0:
        results.append({"gsm": gsm, "model_id": None, "cvcl_id": None,
                         "canonical_name": None, "gse_id": None,
                         "resolution_method": "no_geo_info_entry"})
        continue

    cvcl = row.iloc[0]["cellosaurus_id"]
    gse_id = row.iloc[0]["gse_id"]
    matching_type = row.iloc[0]["matching_type"]
    title = row.iloc[0]["title"]

    sample_match, method = resolve_with_tiebreak(cvcl, geo_title=title)

    if sample_match is not None:
        results.append({
            "gsm": gsm,
            "model_id": sample_match["depmap_id"],
            "cvcl_id": cvcl,
            "canonical_name": sample_match["cell_line_name"],
            "gse_id": gse_id,
            "matching_type": matching_type,
            "resolution_method": method
        })
    else:
        results.append({
            "gsm": gsm, "model_id": None, "cvcl_id": cvcl,
            "canonical_name": None, "gse_id": gse_id,
            "matching_type": matching_type,
            "resolution_method": method  # carries the specific reason through
        })

geo_crosswalk = pd.DataFrame(results)
print(geo_crosswalk["resolution_method"].value_counts(dropna=False))

resolution_method
resolved                                   2293
no_match                                    373
cvcl_exists_but_only_deprecated_ach_ids      14
resolved_via_geo_title                        1
Name: count, dtype: int64


In [120]:
geo_crosswalk["resolution_method"] = geo_crosswalk["resolution_method"].replace(
    "no_match", "geo_only_no_depmap_equivalent"
)

In [121]:
deprecated = geo_crosswalk[
    geo_crosswalk["resolution_method"] == "cvcl_exists_but_only_deprecated_ach_ids"
]
print(deprecated[["gsm", "cvcl_id", "gse_id"]])

             gsm    cvcl_id    gse_id
98    gsm1035324  cvcl_1888      none
1028  gsm1374648  cvcl_0618  gse57083
1029  gsm1374649  cvcl_0618  gse57083
1123  gsm1374743  cvcl_6831  gse57083
1407  gsm1589152  cvcl_0618  gse65216
1527   gsm206489  cvcl_0455      none
1600   gsm219757  cvcl_0041      none
1801   gsm274745  cvcl_0455  gse10843
1802   gsm274746  cvcl_0455  gse10843
1899   gsm274843  cvcl_1888  gse10843
2162   gsm385047  cvcl_1711  gse15329
2178   gsm385063  cvcl_1888  gse15329
2506   gsm844591  cvcl_0618  gse34211
2639   gsm844724  cvcl_1911  gse34211


In [122]:
assert geo_crosswalk["gsm"].is_unique
total = len(geo_crosswalk)
print(f"Total: {total}")
print(f"Resolved (any method): {geo_crosswalk['model_id'].notna().sum()}")

final_crosswalk = geo_crosswalk[geo_crosswalk["model_id"].notna()][
    ["gsm", "model_id", "cvcl_id", "canonical_name", "gse_id", "resolution_method"]
]

no_depmap_or_deprecated = geo_crosswalk[
    geo_crosswalk["resolution_method"].isin(
        ["geo_only_no_depmap_equivalent", "cvcl_exists_but_only_deprecated_ach_ids"]
    )
][["gsm", "cvcl_id", "gse_id", "resolution_method"]]

final_crosswalk.to_csv("geo_expr_cell_line_crosswalk.csv", index=False)
no_depmap_or_deprecated.to_csv("geo_expr_no_depmap_equivalent.csv", index=False)

print(f"Final crosswalk: {len(final_crosswalk)}")
print(f"No DepMap / deprecated: {len(no_depmap_or_deprecated)}")
print(f"Total + excluded earlier (586) = {len(final_crosswalk) + len(no_depmap_or_deprecated) + 586}")
# should equal 3267

Total: 2681
Resolved (any method): 2294
Final crosswalk: 2294
No DepMap / deprecated: 387
Total + excluded earlier (586) = 3267


### Verifying developed crosswalks against the 11 deprecated sample_info IDs

In [123]:
import pandas as pd

# ── Step 1 — rebuild the deprecated ID list fresh ───────────────────
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")

deprecated_mask = sample_info["cellosaurus_issues"].astype(str).str.contains(
    "removed from depmap", case=False, na=False
)
deprecated_ids = set(sample_info[deprecated_mask]["depmap_id"])

print(f"Total deprecated ACH- IDs in sample_info: {len(deprecated_ids)}")  # should be 11
print(deprecated_ids)

Total deprecated ACH- IDs in sample_info: 11
{'ach-000338', 'ach-000109', 'ach-001543', 'ach-002138', 'ach-000621', 'ach-000833', 'ach-002260', 'ach-000428', 'ach-001075', 'ach-001189', 'ach-002392'}


In [124]:
# ── Step 2 — check depmap_expr crosswalk ─────────────────────────────
depmap_crosswalk = pd.read_csv("depmap_expr_cell_line_crosswalk.csv")

depmap_hits = depmap_crosswalk[depmap_crosswalk["model_id"].isin(deprecated_ids)]
print(f"\ndepmap_expr crosswalk rows resting on a deprecated model_id: {len(depmap_hits)}")
if len(depmap_hits) > 0:
    print(depmap_hits[["model_id", "cvcl_id", "canonical_name", "resolution_method"]])


depmap_expr crosswalk rows resting on a deprecated model_id: 7
        model_id    cvcl_id canonical_name   resolution_method
129   ach-000109  cvcl_6831      nci-h3255  sample_info_direct
145   ach-000621  cvcl_0618     mda-mb-157  sample_info_direct
259   ach-000338  cvcl_1711         sr-786  sample_info_direct
608   ach-000428  cvcl_1911          uo-31  sample_info_direct
791   ach-001075  cvcl_0455       nci-h292  sample_info_direct
969   ach-000833  cvcl_0041          rh-30  sample_info_direct
1353  ach-001543  cvcl_1337         kosc-2  sample_info_direct


In [125]:
# ── Step 3 — check hpa_rna crosswalk ─────────────────────────────────
hpa_crosswalk = pd.read_csv("hpa_rna_cell_line_crosswalk.csv")

hpa_hits = hpa_crosswalk[hpa_crosswalk["model_id"].isin(deprecated_ids)]
print(f"\nhpa_rna crosswalk rows resting on a deprecated model_id: {len(hpa_hits)}")
if len(hpa_hits) > 0:
    print(hpa_hits[["hpa_name", "model_id", "cvcl_id", "canonical_name", "resolution_method"]])


hpa_rna crosswalk rows resting on a deprecated model_id: 7
        hpa_name    model_id    cvcl_id canonical_name       resolution_method
527   mda-mb-157  ach-000621  cvcl_0618     mda-mb-157       sample_info_exact
687     nci-h292  ach-001075  cvcl_0455       nci-h292       sample_info_exact
690    nci-h3255  ach-000109  cvcl_6831      nci-h3255       sample_info_exact
826         rh30  ach-000833  cvcl_0041          rh-30       sample_info_exact
848         sc-1  ach-002392  cvcl_1888           sc-1       sample_info_exact
971           sr  ach-000338  cvcl_1711         sr-786  hpa_desc_cvcl_fallback
1079       uo-31  ach-000428  cvcl_1911          uo-31       sample_info_exact


In [126]:
def check_for_active_alternative(cvcl):
    candidates = sample_info[sample_info["rrid"] == cvcl]
    active = candidates[~candidates["cellosaurus_issues"].astype(str).str.contains(
        "removed from depmap", case=False, na=False
    )]
    return active

# Run this for every flagged row in both crosswalks
for cvcl in pd.concat([depmap_hits, hpa_hits])["cvcl_id"].unique():
    print(f"\n--- {cvcl} ---")
    alt = check_for_active_alternative(cvcl)
    print(alt[["depmap_id", "cell_line_name", "cellosaurus_issues"]])


--- cvcl_6831 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_0618 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_1711 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_1911 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_0455 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_0041 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_1337 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []

--- cvcl_1888 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues]
Index: []


In [127]:
depmap_cvcls = set(depmap_hits["cvcl_id"])
hpa_cvcls = set(hpa_hits["cvcl_id"])

shared = depmap_cvcls & hpa_cvcls
depmap_only = depmap_cvcls - hpa_cvcls
hpa_only = hpa_cvcls - depmap_cvcls

print(f"Shared between both crosswalks: {len(shared)} — {shared}")
print(f"depmap_expr only: {depmap_only}")
print(f"hpa_rna only: {hpa_only}")

Shared between both crosswalks: 6 — {'cvcl_0455', 'cvcl_6831', 'cvcl_1911', 'cvcl_0041', 'cvcl_0618', 'cvcl_1711'}
depmap_expr only: {'cvcl_1337'}
hpa_rna only: {'cvcl_1888'}


In [128]:
depmap_profiles = pd.read_parquet("../../data/parquet/data_clean/depmap_profiles_clean.parquet")

# For each deprecated CVCL, check if a DIFFERENT, active ACH- ID
# for the SAME cell line exists in depmap_profiles by name
for cvcl, name in zip(
    ["cvcl_6831","cvcl_0618","cvcl_1711","cvcl_1911","cvcl_0455","cvcl_0041","cvcl_1337","cvcl_1888"],
    ["nci-h3255","mda-mb-157","sr-786","uo-31","nci-h292","rh-30","kosc-2","sc-1"]
):
    # Check Cellosaurus cross-references for a depmap entry not yet in sample_info
    match = cellosaurus[cellosaurus["cellosaurus_accession"] == cvcl]
    if len(match) > 0:
        xref = match.iloc[0]["cross-references"]
        depmap_xref = extract_depmap_id(xref)  # reuse your earlier function
        print(f"{name} ({cvcl}): Cellosaurus cross-ref says depmap ID = {depmap_xref}")

nci-h3255 (cvcl_6831): Cellosaurus cross-ref says depmap ID = ach-000109
mda-mb-157 (cvcl_0618): Cellosaurus cross-ref says depmap ID = ach-000621
sr-786 (cvcl_1711): Cellosaurus cross-ref says depmap ID = ach-000338
uo-31 (cvcl_1911): Cellosaurus cross-ref says depmap ID = ach-000428
nci-h292 (cvcl_0455): Cellosaurus cross-ref says depmap ID = ach-000474
rh-30 (cvcl_0041): Cellosaurus cross-ref says depmap ID = ach-000833
kosc-2 (cvcl_1337): Cellosaurus cross-ref says depmap ID = ach-001543
sc-1 (cvcl_1888): Cellosaurus cross-ref says depmap ID = ach-002303


In [129]:
for ach_id in ["ach-000474", "ach-002303"]:
    print(f"\n--- {ach_id} ---")
    print(sample_info[sample_info["depmap_id"] == ach_id][
        ["depmap_id", "cell_line_name", "cellosaurus_issues", "rrid"]
    ])


--- ach-000474 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues, rrid]
Index: []

--- ach-002303 ---
Empty DataFrame
Columns: [depmap_id, cell_line_name, cellosaurus_issues, rrid]
Index: []


In [130]:
depmap_profiles = pd.read_parquet("../../data/parquet/data_clean/depmap_profiles_clean.parquet")
signatures = pd.read_parquet("../../data/parquet/data_clean/signatures_clean.parquet")

for ach_id in ["ach-000474", "ach-002303"]:
    in_profiles = ach_id in set(depmap_profiles["modelid"])
    in_sigs = ach_id in set(signatures["modelid"])
    print(f"{ach_id}: in depmap_profiles = {in_profiles}, in signatures = {in_sigs}")

ach-000474: in depmap_profiles = False, in signatures = False
ach-002303: in depmap_profiles = False, in signatures = False


In [131]:
remaining_deprecated = {
    "ach-000109": "nci-h3255", "ach-000621": "mda-mb-157", "ach-000338": "sr-786",
    "ach-000428": "uo-31", "ach-000833": "rh-30", "ach-001543": "kosc-2"
}

for ach_id, name in remaining_deprecated.items():
    in_sig = ach_id in set(signatures["modelid"])
    print(f"{name} ({ach_id}): present in signatures = {in_sig}")

nci-h3255 (ach-000109): present in signatures = True
mda-mb-157 (ach-000621): present in signatures = True
sr-786 (ach-000338): present in signatures = True
uo-31 (ach-000428): present in signatures = True
rh-30 (ach-000833): present in signatures = True
kosc-2 (ach-001543): present in signatures = True


In [132]:
deprecated_no_alternative = {"ach-001075", "ach-002392"}  # nci-h292, sc-1 — confirmed nowhere else
deprecated_disputed = {"ach-000109", "ach-000621", "ach-000338",
                       "ach-000428", "ach-000833", "ach-001543"}  # sample_info vs signatures disagree

for df_name, df in [("depmap", depmap_crosswalk), ("hpa", hpa_crosswalk)]:
    mask_no_alt = df["model_id"].isin(deprecated_no_alternative)
    mask_disputed = df["model_id"].isin(deprecated_disputed)

    df.loc[mask_no_alt, "resolution_method"] = "deprecated_confirmed_no_alternative"
    df.loc[mask_disputed, "resolution_method"] = "deprecated_disputed_vs_signatures"

    df.loc[mask_no_alt | mask_disputed, "model_id"] = None

    print(f"{df_name}: moved {mask_no_alt.sum()} confirmed-deprecated, "
          f"{mask_disputed.sum()} disputed-with-signatures")

depmap: moved 1 confirmed-deprecated, 6 disputed-with-signatures
hpa: moved 2 confirmed-deprecated, 5 disputed-with-signatures


In [133]:
depmap_crosswalk[depmap_crosswalk["model_id"].notna()].to_csv(
    "depmap_expr_cell_line_crosswalk.csv", index=False)
depmap_crosswalk[depmap_crosswalk["model_id"].isna()].to_csv(
    "depmap_expr_no_depmap_equivalent.csv", index=False)

hpa_crosswalk[hpa_crosswalk["model_id"].notna()].to_csv(
    "hpa_rna_cell_line_crosswalk.csv", index=False)
hpa_crosswalk[hpa_crosswalk["model_id"].isna()].to_csv(
    "hpa_rna_no_depmap_equivalent.csv", index=False)

In [134]:
print(f"depmap_expr final resolved (model_id not null): {depmap_crosswalk['model_id'].notna().sum()}")
print(f"depmap_expr total moved out: {(depmap_crosswalk['model_id'].isna() & depmap_crosswalk['resolution_method'].isin(['deprecated_confirmed_no_alternative', 'deprecated_disputed_vs_signatures'])).sum()}")

print(f"\nhpa_rna final resolved (model_id not null): {hpa_crosswalk['model_id'].notna().sum()}")
print(f"hpa_rna total moved out: {(hpa_crosswalk['model_id'].isna() & hpa_crosswalk['resolution_method'].isin(['deprecated_confirmed_no_alternative', 'deprecated_disputed_vs_signatures'])).sum()}")

depmap_expr final resolved (model_id not null): 1431
depmap_expr total moved out: 7

hpa_rna final resolved (model_id not null): 1097
hpa_rna total moved out: 7


### New hpa_rna cell line resolution

In [136]:
import pandas as pd
import re

# ── Load sources fresh ───────────────────────────────────────────────
hpa         = pd.read_parquet("../../data/parquet/data_clean/hpa_rna_clean.parquet")
hpa_desc    = pd.read_parquet("../../data/parquet/data_clean/hpa_desc_clean.parquet")
sample_info = pd.read_parquet("../../data/parquet/data_clean/sample_info_clean.parquet")
cellosaurus = pd.read_parquet("../../data/parquet/data_clean/cellosaurus_clean.parquet")

# ── Normalisation helpers (bracket-aware, from earlier fixes) ──────────
def normalise_base(name):
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_+\s]", "", name)
    return name.strip()

def get_bracket_content(name):
    match = re.search(r"\[(.*?)\]", str(name))
    return match.group(1).lower().strip() if match else None

# ── Prep reference fields ───────────────────────────────────────────────
sample_info["cln_norm"]    = sample_info["cell_line_name"].apply(normalise_base)
sample_info["scln_norm"]   = sample_info["stripped_cell_line_name"].apply(normalise_base)
sample_info["rrid_norm"]   = sample_info["rrid"]
sample_info["is_deprecated"] = sample_info["cellosaurus_issues"].astype(str).str.contains(
    "removed from depmap", case=False, na=False
)

hpa_desc["cln_exact"] = hpa_desc["cell line"]  # same-source, exact match — no normalisation needed

cellosaurus["base_norm"]       = cellosaurus["cellosaurus_cell_line_name"].apply(normalise_base)
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

# ── Unique HPA cell lines to resolve ─────────────────────────────────
hpa_lines = hpa[["cell line"]].drop_duplicates().reset_index(drop=True)
hpa_lines["base_norm"]       = hpa_lines["cell line"].apply(normalise_base)
hpa_lines["bracket_content"] = hpa_lines["cell line"].apply(get_bracket_content)

# ── Helper: resolve a CVCL against sample_info, deprecation-aware ──────
def resolve_cvcl_to_model(cvcl):
    if pd.isna(cvcl):
        return None, None, "no_cvcl"

    matches = sample_info[sample_info["rrid_norm"] == cvcl]

    if len(matches) == 0:
        return None, cvcl, "cvcl_not_in_sample_info"

    active = matches[~matches["is_deprecated"]]

    if len(active) == 1:
        row = active.iloc[0]
        return row["depmap_id"], cvcl, "resolved"
    elif len(active) > 1:
        return None, cvcl, "ambiguous_multiple_active_matches"
    else:
        # every candidate deprecated
        return None, cvcl, "cvcl_exists_but_only_deprecated_ach_ids"

# ── Main resolution loop ─────────────────────────────────────────────
results = []

for _, row in hpa_lines.iterrows():
    name, base, bracket = row["cell line"], row["base_norm"], row["bracket_content"]

    # ── TIER 1 — hpa_desc CVCL lookup (same-source, exact name match) ──
    desc_match = hpa_desc[hpa_desc["cln_exact"] == name]
    if len(desc_match) >= 1:
        cvcl = desc_match.iloc[0]["cellosaurus id"]
        if pd.notna(cvcl):
            model_id, cvcl_out, status = resolve_cvcl_to_model(cvcl)
            if model_id is not None:
                results.append({
                    "hpa_name": name, "model_id": model_id, "cvcl_id": cvcl_out,
                    "canonical_name": sample_info.loc[
                        sample_info["depmap_id"] == model_id, "cell_line_name"
                    ].iloc[0],
                    "resolution_method": "hpa_desc_cvcl_tier1"
                })
                continue
            elif status in ("cvcl_exists_but_only_deprecated_ach_ids",
                             "ambiguous_multiple_active_matches"):
                results.append({
                    "hpa_name": name, "model_id": None, "cvcl_id": cvcl_out,
                    "canonical_name": None, "resolution_method": status
                })
                continue
            # else: cvcl_not_in_sample_info — fall through to next tiers

    # ── TIER 2 — sample_info exact name match ───────────────────────────
    m = sample_info[sample_info["cln_norm"] == base]
    active = m[~m["is_deprecated"]]
    if len(active) == 1:
        results.append({
            "hpa_name": name, "model_id": active.iloc[0]["depmap_id"],
            "cvcl_id": active.iloc[0]["rrid"], "canonical_name": active.iloc[0]["cell_line_name"],
            "resolution_method": "sample_info_exact_tier2"
        })
        continue

    # ── TIER 3 — sample_info stripped name match ────────────────────────
    m = sample_info[sample_info["scln_norm"] == base]
    active = m[~m["is_deprecated"]]
    if len(active) == 1:
        results.append({
            "hpa_name": name, "model_id": active.iloc[0]["depmap_id"],
            "cvcl_id": active.iloc[0]["rrid"], "canonical_name": active.iloc[0]["cell_line_name"],
            "resolution_method": "sample_info_stripped_tier3"
        })
        continue

    # ── TIER 4 — Cellosaurus name fallback, disambiguation-aware ────────
    candidates = cellosaurus[cellosaurus["base_norm"] == base]
    chosen = None
    if len(candidates) == 1:
        chosen = candidates.iloc[0]
    elif len(candidates) > 1 and bracket is not None:
        refined = candidates[candidates["bracket_content"] == bracket]
        if len(refined) == 1:
            chosen = refined.iloc[0]

    if chosen is not None:
        cvcl = chosen["cellosaurus_accession"]
        model_id, cvcl_out, status = resolve_cvcl_to_model(cvcl)
        if model_id is not None:
            results.append({
                "hpa_name": name, "model_id": model_id, "cvcl_id": cvcl_out,
                "canonical_name": chosen["cellosaurus_cell_line_name"],
                "resolution_method": "cellosaurus_name_tier4"
            })
        elif status == "cvcl_exists_but_only_deprecated_ach_ids":
            results.append({
                "hpa_name": name, "model_id": None, "cvcl_id": cvcl_out,
                "canonical_name": chosen["cellosaurus_cell_line_name"],
                "resolution_method": "cvcl_exists_but_only_deprecated_ach_ids"
            })
        else:
            results.append({
                "hpa_name": name, "model_id": None, "cvcl_id": cvcl_out,
                "canonical_name": chosen["cellosaurus_cell_line_name"],
                "resolution_method": "hpa_only_no_depmap_equivalent"
            })
        continue

    # ── Genuinely unresolved ─────────────────────────────────────────────
    results.append({
        "hpa_name": name, "model_id": None, "cvcl_id": None,
        "canonical_name": None, "resolution_method": "unresolved"
    })

crosswalk = pd.DataFrame(results)

# ── Validation ────────────────────────────────────────────────────────
assert crosswalk["hpa_name"].is_unique, "Duplicate HPA names in output!"
total = len(crosswalk)
print(crosswalk["resolution_method"].value_counts(dropna=False))
print(f"\nTotal: {total}  (should equal {hpa['cell line'].nunique()})")
print(f"Resolved (has model_id): {crosswalk['model_id'].notna().sum()}")

# ── Split into final outputs ─────────────────────────────────────────
final_crosswalk = crosswalk[crosswalk["model_id"].notna()]
no_depmap_or_deprecated = crosswalk[
    crosswalk["resolution_method"].isin(
        ["hpa_only_no_depmap_equivalent", "cvcl_exists_but_only_deprecated_ach_ids"]
    )
]
ambiguous_or_unresolved = crosswalk[
    crosswalk["resolution_method"].isin(["ambiguous_multiple_active_matches", "unresolved"])
]

final_crosswalk.to_csv("hpa_rna_cell_line_crosswalk1.csv", index=False)
no_depmap_or_deprecated.to_csv("hpa_rna_no_depmap_equivalent.csv1", index=False)
ambiguous_or_unresolved.to_csv("unmapped_hpa_rna.csv11", index=False)

print(f"\nFinal crosswalk: {len(final_crosswalk)}")
print(f"No DepMap / deprecated: {len(no_depmap_or_deprecated)}")
print(f"Ambiguous / unresolved: {len(ambiguous_or_unresolved)}")
print(f"Reconciliation: {len(final_crosswalk) + len(no_depmap_or_deprecated) + len(ambiguous_or_unresolved)} / {total}")

resolution_method
hpa_desc_cvcl_tier1                        1097
hpa_only_no_depmap_equivalent                94
ambiguous_multiple_active_matches             8
cvcl_exists_but_only_deprecated_ach_ids       7
Name: count, dtype: int64

Total: 1206  (should equal 1206)
Resolved (has model_id): 1097

Final crosswalk: 1097
No DepMap / deprecated: 101
Ambiguous / unresolved: 8
Reconciliation: 1206 / 1206


# Just checks

In [135]:
hpa_rna_names = set(hpa["cell line"].unique())
hpa_desc_names = set(hpa_desc["cell line"].unique())

direct_overlap = hpa_rna_names & hpa_desc_names
print(f"hpa_rna unique names: {len(hpa_rna_names)}")
print(f"hpa_desc unique names: {len(hpa_desc_names)}")
print(f"Direct overlap (same spelling): {len(direct_overlap)}")

hpa_rna unique names: 1206
hpa_desc unique names: 1206
Direct overlap (same spelling): 1206


In [138]:
geo_info = pd.read_parquet("../../data/parquet/data_clean/geo_info_clean.parquet")
geo_info["cellosaurus_id"].isna().sum()

0

In [139]:
cvcl_7082_rows = geo_info[geo_info["cellosaurus_id"] == "cvcl_7082"]
print(cvcl_7082_rows[["geo_accession", "cellline", "gse_id", "matching_type"]].head())

    geo_accession cellline gse_id matching_type
83     gsm1035304     none   none          none
85     gsm1035306     none   none          none
97     gsm1035318     none   none          none
98     gsm1035319     none   none          none
102    gsm1035323     none   none          none


In [140]:
depmap.head(2)

,ensg00000000003,ensg00000000005,ensg00000000419,ensg00000000457,ensg00000000460,ensg00000000938,ensg00000000971,ensg00000001036,ensg00000001084,ensg00000001167,...,ensg00000288714,ensg00000288717,ensg00000288718,ensg00000288719,ensg00000288720,ensg00000288721,ensg00000288722,ensg00000288723,ensg00000288724,ensg00000288725
pr-adbjpg,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,4.739848,...,0.0,0.536053,0.0,0.028569,0.176323,0.992768,2.797013,0.000000,0.0,0.000000
pr-i2azwg,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,3.481557,...,0.0,0.879706,0.0,0.014355,0.014355,0.432959,2.972693,0.056584,0.0,0.070389
